# Лабораторна. BookScout: збираємо дані з вебсайту

**Тривалість:** 60 хвилин

**Контекст.** Ви — аналітик невеликого стартапу BookScout, який допомагає читачам знаходити книги за найкращою ціною. Керівник просить зібрати дані про всі книги інтернет-магазину й відповісти на кілька питань: скільки коштують книги та чи дорожчі книги з вищим рейтингом.

У магазину немає готового файлу з даними, тому ми зберемо їх просто з вебсторінок. Це називається **web scraping** (скрейпінг, парсинг сайтів).

**Сайт:** https://books.toscrape.com — навчальний магазин, створений спеціально для практики (1000 книг, 50 сторінок, по 20 книг на сторінці).

**План роботи:**

| Частина | Що робимо | Час |
|---|---|---|
| 0 | Підготовка | 3 хв |
| 1 | Як влаштована вебсторінка + тренування на маленькому прикладі | 10 хв |
| 2 | Перший запит до сайту | 7 хв |
| 3 | Дістаємо дані про одну книгу | 15 хв |
| 4 | Збираємо всі сторінки | 8 хв |
| 5 | Аналізуємо дані на чистому Python | 15 хв |
| 6 | Підсумок | 2 хв |

**Що здати:** цей ноутбук із заповненим кодом і відповідями в комірках «Ваша відповідь», а також файл `books.csv`.

**Оцінювання:** максимум 1.25 бала. Бали вказано біля кожного завдання. Якщо кілька завдань оцінюються разом, бали ставляться, лише коли виконано всі з них. Код, що падає з помилкою, і відповіді без пояснення не зараховуються.

## Частина 0. Підготовка (3 хв)

Нам знадобляться дві бібліотеки та один вбудований модуль:
- `requests` — «ходить» на сайт і завантажує сторінку, як браузер;
- `BeautifulSoup` (з пакета `beautifulsoup4`) — допомагає знайти потрібні шматки в HTML-коді сторінки;
- `csv` — вбудований модуль Python для збереження даних у файл-таблицю (встановлювати не треба).

Якщо бібліотеки не встановлені, розкоментуйте перший рядок (приберіть `#`) і виконайте комірку.

In [31]:
# !pip install requests beautifulsoup4

import csv
import time
import requests
from bs4 import BeautifulSoup

print("Усе готово!")

Усе готово!


## Частина 1. Як влаштована вебсторінка (10 хв)

### Що відбувається, коли ви відкриваєте сайт

1. Браузер надсилає **запит** (request) на сервер: «дай мені сторінку за цією адресою».
2. Сервер повертає **відповідь** (response) — текст у форматі **HTML**.
3. Браузер «малює» цей HTML: кнопки, картинки, таблиці.

Скрейпер робить кроки 1–2 так само, але замість малювання **читає HTML як текст** і витягує з нього потрібні дані.

### Що таке HTML

HTML складається з **тегів**. Тег має назву, може мати **атрибути** і містити текст або інші теги:

```html
<p class="price">25.50 UAH</p>
```

- `p` — назва тегу (абзац);
- `class="price"` — атрибут `class` зі значенням `price`. Класи — це «мітки», за якими зручно шукати потрібні елементи;
- `25.50 UAH` — текст усередині тегу.

Теги вкладаються один в одного, як коробки в коробках. Нижче — маленька «сторінка» уявного фруктового магазину. Подивіться на неї уважно: справжній книжковий магазин влаштований дуже схоже.

In [32]:
mini_html = """
<div class="shop">
  <div class="item">
    <h3><a href="apple.html" title="Green Apple from Poltava">Green Apple...</a></h3>
    <p class="stars Four"></p>
    <p class="price">25.50 UAH</p>
  </div>
  <div class="item">
    <h3><a href="pear.html" title="Sweet Pear">Sweet Pear</a></h3>
    <p class="stars Two"></p>
    <p class="price">40.00 UAH</p>
  </div>
  <div class="item">
    <h3><a href="plum.html" title="Blue Plum from Vinnytsia">Blue Plum...</a></h3>
    <p class="stars Five"></p>
    <p class="price">18.75 UAH</p>
  </div>
</div>
"""

### Як шукати елементи з BeautifulSoup

Спочатку перетворюємо HTML-текст на об'єкт `soup`, у якому зручно шукати:

```python
soup = BeautifulSoup(mini_html, "html.parser")
```

Шукаємо елементи за допомогою **CSS-селекторів** — коротких «адрес» елементів:

| Селектор | Що знаходить |
|---|---|
| `"p"` | усі теги `<p>` |
| `".price"` | усі елементи з класом `price` |
| `"p.price"` | теги `<p>` з класом `price` |
| `"div.item"` | теги `<div>` з класом `item` |
| `"h3 a"` | теги `<a>`, які лежать **всередині** `<h3>` |

Два головні методи:
- `soup.select("селектор")` — повертає **список** усіх знайдених елементів;
- `soup.select_one("селектор")` — повертає **перший** знайдений елемент (або `None`, якщо нічого немає).

Що можна дістати зі знайденого елемента:
- `.text` — текст усередині тегу (часто з зайвими пробілами, тому додаємо `.strip()`);
- `["назва_атрибута"]` — значення атрибута, наприклад `link["title"]` або `link["href"]`;
- `.get("class")` — список класів елемента, наприклад `["stars", "Four"]`.

Виконайте приклад і уважно прочитайте, що він друкує:

In [33]:
soup_mini = BeautifulSoup(mini_html, "html.parser")

items = soup_mini.select("div.item")      # список з 3 елементів
print("Кількість товарів:", len(items))

first_item = items[0]                     # працюємо лише з першим товаром

price_tag = first_item.select_one("p.price")
print("Тег ціни:", price_tag)
print("Текст ціни:", price_tag.text)

link = first_item.select_one("h3 a")
print("Видимий текст посилання:", link.text)
print("Атрибут title:", link["title"])

stars = first_item.select_one("p.stars")
print("Класи зірочок:", stars.get("class"))

Кількість товарів: 3
Тег ціни: <p class="price">25.50 UAH</p>
Текст ціни: 25.50 UAH
Видимий текст посилання: Green Apple...
Атрибут title: Green Apple from Poltava
Класи зірочок: ['stars', 'Four']


**Питання 1.1.** *(0.25 б. за 1.1–2.3)* Чому видимий текст посилання (`link.text`) і атрибут `title` відрізняються? Який із них краще використовувати, щоб отримати **повну** назву товару?

**Ваша відповідь:** 
`link.text` дає нам текст в тілі, той який ми бачемо на екрані на сторінці і він може бути
обрізаний для краси
`link["title"]` дає там текст атрибуту, який є фактичною назвою цього товару без візульних скорочень, з яким працює код для ідентифікації (моя підозра)
Тому для повної фактичної назви краще використовувати `link["title"]`


**Завдання 1.2.** *(0.25 б. за 1.1–2.3)* Пройдіться циклом `for` по списку `items` і для кожного товару виведіть його повну назву та ціну (текстом, як є). Очікуваний результат:

```
Green Apple from Poltava 25.50 UAH
Sweet Pear 40.00 UAH
Blue Plum from Vinnytsia 18.75 UAH
```

Кроки:
1. Напишіть цикл `for item in items:`.
2. Усередині циклу знайдіть у `item` посилання так само, як у прикладі вище, і візьміть його атрибут `title`.
3. Так само знайдіть у `item` тег ціни й візьміть його текст.
4. Виведіть обидва значення через `print`.

In [34]:
# TODO
for item in items:
    name = item.select_one("h3 a")["title"]
    price = item.select_one("p.price").text
    print(name, price)

Green Apple from Poltava 25.50 UAH
Sweet Pear 40.00 UAH
Blue Plum from Vinnytsia 18.75 UAH


## Частина 2. Перший запит до справжнього сайту (7 хв)

Тепер зробимо те саме зі справжньою сторінкою. Функція `requests.get(адреса)` завантажує сторінку й повертає об'єкт-відповідь:
- `response.status_code` — код результату (200 означає «усе добре»);
- `response.text` — HTML-код сторінки як рядок.

Параметр `timeout=10` означає «не чекати відповіді довше 10 секунд», щоб програма не зависла.

Виконайте код:

In [35]:
url = "https://books.toscrape.com/catalogue/page-1.html"

response = requests.get(url, timeout=10)
response.encoding = "utf-8"   # пояснення нижче

print("Код відповіді:", response.status_code)
print(response.text[:500])    # перші 500 символів HTML

soup = BeautifulSoup(response.text, "html.parser")
print(soup.select_one("p.price_color"))

Код відповіді: 200


<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8"
<p class="price_color">£51.77</p>


**Навіщо рядок `response.encoding = "utf-8"`?** Текст на комп'ютері зберігається як числа, а **кодування** — це правило, яке число відповідає якій літері. Сторінка збережена в кодуванні UTF-8, але `requests` іноді вгадує неправильно. Тоді символ фунта `£` перетворюється на `Â£`. Цей рядок явно каже: «читай як UTF-8».

**Питання 2.1.** *(0.25 б. за 1.1–2.3)* Закоментуйте рядок з `encoding`, виконайте комірку ще раз і знайдіть у виведеному тексті будь-яку ціну (пошукайте `price_color`). Що змінилося? Поверніть рядок назад.

**Ваша відповідь:** 
Знак фунту змінився з `£` на `Â£` як і було в пояснені

**Питання 2.2.** *(0.25 б. за 1.1–2.3)* Відкрийте https://books.toscrape.com/catalogue/page-1.html у браузері. Натисніть правою кнопкою миші на назву будь-якої книги → **Inspect** («Переглянути код»). Відкриється панель з HTML-кодом. Знайдіть:
- тег і клас, який обгортає **одну книгу** цілком (підказка: тег `article`);
- де лежить повна назва книги;
- тег і клас ціни;
- тег, у класі якого записано рейтинг.

Порівняйте з фруктовим магазином з Частини 1: що на що схоже?

**Ваша відповідь:** 
- тег: `article`, клас: `product_pod`
- в атрибуті `title` в тегу посилання
- тег: `з`, клас: `price_color`
- тег: `p`

Схожі тайтли, схожі ціни

**Завдання 2.3.** *(0.25 б. за 1.1–2.3)* Створіть `soup` зі справжньої сторінки й порахуйте, скільки на ній книг. Очікуваний результат: 20.

Кроки:
1. Створіть `BeautifulSoup` з `response.text` так само, як у Частині 1.
2. Знайдіть усі книги методом `select` із селектором `"article.product_pod"`.
3. Виведіть кількість знайдених елементів через `len`.

In [36]:
soup = BeautifulSoup(response.text, "html.parser")
books_on_page = soup.select("article.product_pod")
print(len(books_on_page))

20


## Частина 3. Дістаємо дані про одну книгу (15 хв)

Зараз напишемо **функцію**, яка отримує HTML-елемент однієї книги й повертає словник з її даними. Словник зручний тим, що потім із багатьох словників легко зробити таблицю.

Для кожної книги нам потрібні:

| ключ | тип | приклад |
|---|---|---|
| `title` | str | `"A Light in the Attic"` |
| `price` | float | `51.77` |
| `rating` | int (1–5) | `3` |

Розберемо кожне поле окремо.

**Назва.** Лежить в атрибуті `title` посилання `h3 a` — так само, як у фруктовому магазині. Цю частину ми вже написали за вас у шаблоні нижче.

**Ціна.** Текст ціни виглядає як `"£51.77"`. Щоб рахувати середню ціну, потрібне **число**, а не рядок. Тому:
1. приберіть символ `£` методом `.replace("£", "")`;
2. перетворіть рядок, що залишився, на число функцією `float()`.

**Рейтинг.** Рейтинг на сайті показаний зірочками-картинками, а в HTML записаний **словом у класі**: `<p class="star-rating Three">`. Метод `.get("class")` поверне список `["star-rating", "Three"]`. Слово рейтингу — другий елемент списку (індекс `1`). Щоб перетворити слово на число, використайте готовий словник `RATING_MAP`: `RATING_MAP["Three"]` дає `3`.

In [42]:
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

first_book = books_on_page[0]

# Назва — готовий приклад
title = first_book.select_one("h3 a")["title"]
print(title)

def book_to_dict(book):
    title = book.select_one("h3 a")["title"]
    price = float(book.select_one("p.price_color").text.replace("£", ""))
    rating = RATING_MAP[book.select_one("p.star-rating").get("class")[1]]

    dic = {
        "title": title,
        "price": price,
        "rating": rating
    }

    return dic

print(book_to_dict(first_book))
    

A Light in the Attic
{'title': 'A Light in the Attic', 'price': 51.77, 'rating': 3}


**Завдання 3.1.** *(0.25 б. за 3.1–3.3)* Для `first_book` отримайте ціну як число.

Кроки:
1. Знайдіть у `first_book` тег з селектором `"p.price_color"` і візьміть його текст.
2. Приберіть із тексту символ `£`.
3. Перетворіть результат на `float` і збережіть у змінну `price`.
4. Виведіть `price` і `type(price)`. Має бути `51.77` і `<class 'float'>`.

In [44]:
# TODO
price = float(first_book.select_one("p.price_color").text.replace("£", ""))
print(price, type(price))

51.77 <class 'float'>


**Завдання 3.2.** *(0.25 б. за 3.1–3.3)* Для `first_book` отримайте рейтинг як число.

Кроки:
1. Знайдіть у `first_book` тег з селектором `"p.star-rating"`.
2. Отримайте список його класів через `.get("class")` і виведіть його, щоб побачити, як він виглядає.
3. Візьміть другий елемент списку (слово рейтингу).
4. Перетворіть слово на число через `RATING_MAP` і збережіть у змінну `rating`. Має вийти `3`.

In [46]:
# TODO
rating = RATING_MAP[first_book.select_one("p.star-rating").get("class")[1]]
print(rating)

3


**Завдання 3.3.** *(0.25 б. за 3.1–3.3)* Тепер зберіть усе разом у функцію `parse_book(book)`. Вона приймає елемент однієї книги й повертає словник з ключами `title`, `price`, `rating`.

Кроки:
1. Перенесіть у функцію код отримання назви, ціни та рейтингу з попередніх комірок, замінивши `first_book` на параметр `book`.
2. Поверніть словник `{"title": ..., "price": ..., "rating": ...}`.
3. Перевірте функцію на першій і на п'ятій книзі сторінки.

In [47]:
def parse_book(book):
    title = book.select_one("h3 a")["title"]
    price = float(book.select_one("p.price_color").text.replace("£", ""))
    rating = RATING_MAP[book.select_one("p.star-rating").get("class")[1]]

    dic = {
        "title": title,
        "price": price,
        "rating": rating
    }

    return dic

print(parse_book(books_on_page[0]))
print(parse_book(books_on_page[4]))

{'title': 'A Light in the Attic', 'price': 51.77, 'rating': 3}
{'title': 'Sapiens: A Brief History of Humankind', 'price': 54.23, 'rating': 5}


## Частина 4. Збираємо всі сторінки (8 хв)

Книги розкладені по 50 сторінках, по 20 книг на кожній. Щоб зібрати всі книги, програма має по черзі пройти всі сторінки. Це називається **пагінація**, і є два поширені способи її обійти.

**Спосіб 1. Кнопка «next».** Унизу кожної сторінки, крім останньої, є кнопка «next» з посиланням на наступну сторінку. Програма може поводитися як людина: зібрати книги, знайти кнопку «next», перейти за її посиланням і повторювати це, доки кнопка не зникне. Перевага: не потрібно знати, скільки всього сторінок.

**Спосіб 2. Номер сторінки в адресі.** Подивіться на адреси сторінок каталогу:

```
https://books.toscrape.com/catalogue/page-1.html
https://books.toscrape.com/catalogue/page-2.html
...
https://books.toscrape.com/catalogue/page-50.html
```

Змінюється тільки число, решта адреси однакова. Тому можна просто **підставляти номер сторінки в адресу** в циклі, не шукаючи кнопок. Перевага: дуже простий код. Недолік: треба знати, скільки сторінок (або перевіряти, коли сторінки закінчились).

У лабораторній використаємо **спосіб 2**, а спосіб 1 залишимо для бонусу.

У Python адресу зручно зібрати f-рядком, підставивши номер у фігурні дужки. Виконайте приклад:

In [ ]:
for n in range(1, 4):
    page_url = f"https://books.toscrape.com/catalogue/page-{n}.html"
    print(page_url)

**Етика скрейпінгу.** Сервер сайту обслуговує й інших людей. Якщо надсилати сотні запитів за секунду, сайт може «лягти» або заблокувати вас. Тому між запитами робимо паузу `time.sleep(0.5)` — пів секунди очікування.

Нижче — готова функція, яка завантажує **одну** сторінку й повертає список словників для всіх книг на ній. Прочитайте її: вона складається з кроків, які ви вже робили.

In [ ]:
def parse_page(url):
    response = requests.get(url, timeout=10)
    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "html.parser")
    books = soup.select("article.product_pod")
    return [parse_book(book) for book in books]

print(len(parse_page("https://books.toscrape.com/catalogue/page-2.html")))

**Завдання 4.1.** *(0.25 б. за 4.1 + 4.2)* Напишіть функцію `scrape_catalogue(pages)`, яка збирає книги з перших `pages` сторінок в **один** список.

Кроки:
1. Створіть порожній список `all_books`.
2. Напишіть цикл по номерах сторінок від 1 до `pages` включно (згадайте, як працює `range`).
3. Для кожного номера підставте його в адресу f-рядком (як у прикладі вище), викличте `parse_page` і додайте отримані книги до `all_books`. Увага: `parse_page` повертає список, тож використайте `.extend()`, а не `.append()`.
4. Виведіть повідомлення про прогрес, наприклад `Сторінка 3 готова`.
5. Зробіть паузу `time.sleep(0.5)`.
6. Після циклу поверніть `all_books`.

Спочатку перевірте функцію на 2 сторінках (має бути 40 книг), і лише потім запускайте на всіх 50.

In [ ]:
def scrape_catalogue(pages):
    # TODO
    ...

test_books = scrape_catalogue(2)
print(len(test_books))

In [ ]:
# Повний збір: ~30–40 секунд. Запускайте тільки після успішного тесту вище.
all_books = scrape_catalogue(50)
print(len(all_books))

**Питання 4.2.** *(0.25 б. за 4.1 + 4.2)* Що буде, якщо замість `.extend()` використати `.append()`? Скільки елементів буде в `all_books` і що кожен із них являтиме собою?

**Ваша відповідь:** 

## Частина 5. Аналізуємо дані на чистому Python (15 хв)

Тепер у нас є список `all_books` з 1000 словників, і кожен словник — це одна книга:
```python
[{"title": "A Light in the Attic", "price": 51.77, "rating": 3}, {...}, ...]
```
Щоб відповісти на питання керівника, не потрібні нові бібліотеки: вистачить списків, словників і вбудованих функцій Python. Спершу потренуємося на маленькому списку фруктів, а потім зробимо те саме з книгами.

In [ ]:
fruits = [
    {"title": "Green Apple", "price": 25.5,  "rating": 4},
    {"title": "Sweet Pear",  "price": 40.0,  "rating": 2},
    {"title": "Blue Plum",   "price": 18.75, "rating": 5},
    {"title": "Red Apple",   "price": 30.0,  "rating": 4},
]

### Прийом 1. Список значень одного ключа

Щоб порахувати середню ціну, спершу зберемо всі ціни в окремий список через list comprehension: «для кожного фрукта візьми його ціну».
```python
prices = [fruit["price"] for fruit in fruits]   # [25.5, 40.0, 18.75, 30.0]
```
Далі працюють вбудовані функції: `sum(prices)` — сума, `len(prices)` — кількість, `min(prices)` і `max(prices)` — найменше й найбільше значення. Середнє — це `sum(prices) / len(prices)`.

### Прийом 2. Найдорожчий елемент цілком

`max(prices)` поверне лише **число** 40.0, а не назву фрукта. Щоб отримати весь словник, передамо в `max` сам список словників і параметр `key` — функцію, яка каже, **за чим** порівнювати:
```python
def get_price(item):
    return item["price"]

max(fruits, key=get_price)   # {'title': 'Sweet Pear', 'price': 40.0, 'rating': 2}
```
Зверніть увагу: `get_price` пишемо **без дужок**, бо передаємо саму функцію, а `max` викличе її для кожного елемента.

### Прийом 3. Підрахунок у словнику

Щоб порахувати, скільки фруктів має кожен рейтинг, заводимо порожній словник і для кожного фрукта збільшуємо лічильник його рейтингу. Метод `counts.get(ключ, 0)` повертає поточне значення або 0, якщо ключа ще немає:
```python
counts = {}
for fruit in fruits:
    counts[fruit["rating"]] = counts.get(fruit["rating"], 0) + 1
# {4: 2, 2: 1, 5: 1}
```

### Прийом 4. Групування: словник зі списками

Щоб порахувати середню ціну **для кожного рейтингу**, спершу розкладаємо ціни «по кошиках»: ключ — рейтинг, значення — список цін фруктів з цим рейтингом. Метод `setdefault(ключ, [])` створює порожній список, якщо ключа ще немає, і повертає його, тож до нього одразу можна додати ціну:
```python
groups = {}
for fruit in fruits:
    groups.setdefault(fruit["rating"], []).append(fruit["price"])
# {4: [25.5, 30.0], 2: [40.0], 5: [18.75]}
```
Потім для кожного кошика рахуємо середнє.

### Прийом 5. Фільтр і сортування

- **Фільтр** — list comprehension з умовою: `[f for f in fruits if f["rating"] >= 4]` залишає лише фрукти з рейтингом від 4.
- **Сортування** — `sorted(список, key=get_price, reverse=True)` повертає новий список, відсортований за ціною від більшої до меншої (`reverse=True`). Перші N елементів — це зріз `[:N]`.

Виконайте приклади й подивіться на результат кожного:

In [ ]:
def get_price(item):
    return item["price"]

prices = [fruit["price"] for fruit in fruits]
print("Ціни:", prices)
print("Середня:", sum(prices) / len(prices))
print("Найдорожчий:", max(fruits, key=get_price))

counts = {}
for fruit in fruits:
    counts[fruit["rating"]] = counts.get(fruit["rating"], 0) + 1
print("Кількість за рейтингом:", counts)

groups = {}
for fruit in fruits:
    groups.setdefault(fruit["rating"], []).append(fruit["price"])
for rating in sorted(groups):
    print("Рейтинг", rating, "-> середня ціна", sum(groups[rating]) / len(groups[rating]))

good = [f for f in fruits if f["rating"] >= 4]
print("Топ-2 найдорожчих з рейтингом від 4:", sorted(good, key=get_price, reverse=True)[:2])

Тепер працюємо зі справжніми даними — списком `all_books`.

**Завдання 5.1.** *(0.25 б. за 5.1–5.3)* Перевірте, що у списку 1000 книг. Зберіть усі ціни в список `prices` і виведіть середню ціну (округлену до двох знаків функцією `round(число, 2)`), мінімальну та максимальну.

In [ ]:
# TODO

**Ваша відповідь:** 

**Завдання 5.2.** *(0.25 б. за 5.1–5.3)* Виведіть назву й ціну найдорожчої та найдешевшої книги (Прийом 2).

In [ ]:
# TODO

**Завдання 5.3.** *(0.25 б. за 5.1–5.3)* Порахуйте, скільки книг має кожен рейтинг від 1 до 5 (Прийом 3). Виведіть результат так, щоб рейтинги йшли по порядку, наприклад:
```
Рейтинг 1: ... книг
Рейтинг 2: ... книг
```
Підказка: `sorted(counts)` повертає відсортований список ключів словника. Яких книг найбільше?

In [ ]:
# TODO

**Ваша відповідь:** 

**Завдання 5.4.** *(0.25 б. за 5.4–5.6 + 6.1–6.2)* Порахуйте середню ціну книги для кожного рейтингу (Прийом 4) і виведіть результат по порядку рейтингів з округленням до двох знаків.

Це головне питання керівника: **чи дорожчі книги з вищим рейтингом?** Подивіться на результат і дайте відповідь одним-двома реченнями.

In [ ]:
# TODO

**Ваша відповідь:** 

**Завдання 5.5.** *(0.25 б. за 5.4–5.6 + 6.1–6.2)* Знайдіть 5 найдорожчих книг серед книг з рейтингом 5.

Кроки:
1. Відфільтруйте список, залишивши лише книги з рейтингом 5.
2. Відсортуйте результат за ціною від більшої до меншої.
3. Візьміть перші 5 книг і виведіть назву й ціну кожної.

In [ ]:
# TODO

**Завдання 5.6.** *(0.25 б. за 5.4–5.6 + 6.1–6.2)* Збережіть усі книги у файл `books.csv`.

CSV — це текстовий формат таблиці: кожен рядок — запис, значення розділені комами. Його відкриває Excel, Google Sheets і будь-яка мова програмування. Для запису списку словників у Python є модуль `csv`:
```python
with open("books.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["title", "price", "rating"])
    writer.writeheader()          # перший рядок — назви колонок
    writer.writerows(all_books)   # по рядку на кожен словник
```
- `fieldnames` — назви колонок; вони мають збігатися з ключами словників;
- `newline=""` потрібен, щоб на Windows не з'являлися порожні рядки між записами;
- `encoding="utf-8"` — щоб коректно зберігалися будь-які символи в назвах.

Збережіть файл, а потім відкрийте його (в Jupyter, Excel або текстовому редакторі) і переконайтеся, що в ньому 1001 рядок: заголовок і 1000 книг.

In [ ]:
# TODO

## Частина 6. Підсумок (2 хв) · 0.25 б. разом із 5.4–5.6

**6.1.** Уявіть, що завтра магазин перейменував клас `price_color` на `product-price`. Який рядок вашого коду зламається і яку помилку ви, найімовірніше, побачите?

**6.2.** Чому під час скрейпінгу ми робимо паузи між запитами?

**Ваша відповідь:** 

## Бонус (якщо залишився час) · не оцінюється

**Б.1.** Зараз ми знаємо, що сторінок рівно 50. Але магазин може додати книги. На кожній сторінці, крім останньої, є кнопка «next» (селектор `"li.next a"`). Реалізуйте спосіб 1 з Частини 4: перепишіть `scrape_catalogue` так, щоб вона йшла сторінками, **доки кнопка «next» існує**, і не потребувала числа сторінок. Підказка: використайте цикл `while`, а `select_one` повертає `None`, якщо елемента немає.

**Б.2.** Розподіліть книги за ціновими категоріями: «дешеві» (до £20), «середні» (від £20 до £40) і «дорогі» (від £40). Порахуйте, скільки книг у кожній категорії, і виведіть результат у вигляді простої текстової діаграми, де кожна `#` — це 10 книг, наприклад:
```
дешеві  ##########
середні ##################
дорогі  #######
```
Як розподілені ціни: більшість книг дешеві, дорогі чи ціни розкидані рівномірно?

In [ ]:
# Бонус